# 02 - Training and Testing YOLOv9-C
training yolov9-c on our vehicle images for 10 epochs, checking how it performs on the test set, and looking at predictions

In [ ]:
# colab setup (skip if running locally)
import os

if os.path.exists("/content"):
    if not os.path.exists("/content/RoadWatcher"):
        !git clone https://github.com/adamaarbouba/RoadWatcher.git /content/RoadWatcher
    else:
        !git -C /content/RoadWatcher pull
    os.chdir("/content/RoadWatcher/notebooks")

print("Working directory:", os.getcwd())

In [ ]:
# imports and check gpu
import os
import sys
import urllib.request
from pathlib import Path
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU device:     ", torch.cuda.get_device_name(0))
else:
    print("Warning: CUDA is not available. Please connect a GPU runtime in Colab.")

In [ ]:
# install yolo dependencies
!pip install -q -r ../yolov9/requirements.txt

In [ ]:
# set up folders and cd into yolov9
project_dir = Path.cwd().parent
yolo_dir = project_dir / "yolov9"
data_yaml = project_dir / "roadwatcher.yaml"
data_dir = project_dir / "data_clean"

print("Project folder: ", project_dir)
print("YOLO directory: ", yolo_dir)
print("Data YAML:      ", data_yaml)

assert data_yaml.exists(), "roadwatcher.yaml not found! Run 01_prepare_data.ipynb first."

# quick patch for numpy 2.0 so trapz doesn't crash
metrics_path = yolo_dir / "utils" / "metrics.py"
if metrics_path.exists():
    m_txt = metrics_path.read_text(encoding="utf-8")
    if "np.trapz(" in m_txt:
        m_txt = m_txt.replace("ap = np.trapz(np.interp(x, mrec, mpre), x)", "trapz_fn = getattr(np, 'trapezoid', getattr(np, 'trapz', None))\n        ap = trapz_fn(np.interp(x, mrec, mpre), x)")
        metrics_path.write_text(m_txt, encoding="utf-8")

# cd into yolov9 so scripts run cleanly
os.chdir(yolo_dir)
print("Current directory:", Path.cwd())


In [ ]:
# download pretrained yolov9-c weights if we don't have them
weights_file = Path("yolov9-c.pt")

if not weights_file.exists():
    print("Downloading yolov9-c.pt pretrained weights...")
    url = "https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt"
    urllib.request.urlretrieve(url, "yolov9-c.pt")
    print("Download finished!")
else:
    print("Pretrained weights already exist:", weights_file.resolve())

## Training YOLOv9-C
settings from the project brief:
- yolov9-c architecture (`models/detect/yolov9-c.yaml`)
- yolov9-c.pt pretrained weights
- 640 image size, batch size 4, 10 epochs
- scratch-high hyperparameters

In [ ]:
# start training (10 epochs)
!python train_dual.py \
    --device 0 \
    --workers 2 \
    --img 640 \
    --batch-size 4 \
    --epochs 10 \
    --data ../roadwatcher.yaml \
    --cfg models/detect/yolov9-c.yaml \
    --weights yolov9-c.pt \
    --hyp data/hyps/hyp.scratch-high.yaml \
    --name roadwatcher_yolov9c \
    --min-items 0 \
    --close-mosaic 2 \
    --exist-ok

In [ ]:
# check if weights got saved and display results.png
from IPython.display import Image as DisplayImage, display

best_pt = Path("runs/train/roadwatcher_yolov9c/weights/best.pt")
last_pt = Path("runs/train/roadwatcher_yolov9c/weights/last.pt")

print("best.pt exists:", best_pt.exists())
print("last.pt exists:", last_pt.exists())

results_png = Path("runs/train/roadwatcher_yolov9c/results.png")
if results_png.exists():
    display(DisplayImage(filename=str(results_png)))
else:
    print("results.png not found yet (will show up after training finishes)")

## Testing on the Test Set
test our best model on the 405 test pictures with 0.001 confidence to calculate map

In [ ]:
# evaluate on the 405 test images
!python val_dual.py \
    --data ../roadwatcher.yaml \
    --weights runs/train/roadwatcher_yolov9c/weights/best.pt \
    --img 640 \
    --batch-size 4 \
    --conf-thres 0.001 \
    --iou-thres 0.7 \
    --task test \
    --verbose \
    --exist-ok

## Visual Predictions
run detection with 0.25 confidence so boxes look clean

In [ ]:
# run detector on test images
!python detect_dual.py \
    --weights runs/train/roadwatcher_yolov9c/weights/best.pt \
    --source ../data_clean/test/images \
    --data ../roadwatcher.yaml \
    --img 640 \
    --conf-thres 0.25 \
    --iou-thres 0.45 \
    --project runs/detect \
    --name roadwatcher_test \
    --exist-ok

In [ ]:
# show the first 6 predictions
pred_dir = Path("runs/detect/roadwatcher_test")
pred_images = sorted(list(pred_dir.glob("*.jpg")) + list(pred_dir.glob("*.png")))

print("Total prediction images generated:", len(pred_images))
print("Displaying first 6 predictions:")

for img_path in pred_images[:6]:
    print(img_path.name)
    display(DisplayImage(filename=str(img_path), width=500))

## What to Download
grab these files before colab shuts down:
1. `best.pt` and `last.pt` weights
2. `results.png` loss curves
3. some sample detection images from `runs/detect/roadwatcher_test/`

In [ ]:
# save best.pt to google drive so we don't lose it
from google.colab import drive
drive.mount("/content/drive")

import shutil
from pathlib import Path

best_pt = Path("runs/train/roadwatcher_yolov9c/weights/best.pt")
drive_dest = Path("/content/drive/MyDrive/RoadWatcher")
drive_dest.mkdir(parents=True, exist_ok=True)

if best_pt.exists():
    shutil.copy2(str(best_pt), str(drive_dest / "best.pt"))
    print("Copied best.pt to Google Drive:", drive_dest / "best.pt")
else:
    print("best.pt not found. Make sure training completed successfully.")